In [40]:
import pandas as pd
import numpy as np

In [41]:
data_df = pd.read_csv("yolo_data/all_csv_files/combined_dataset_info_v5.csv")

In [42]:
CG_data_df = data_df[(data_df["dataset"] == "CG") & (data_df["train_or_test"] == "test")]

In [43]:
CG_data_df

,dataset,patient_ID,file_name,nodule_type,train_or_test
8505,CG,5361EE8C3C1F32805CF7DCD382C3BBF132CEBF26,d391e75a9b981397e84339c885c864da_left,single,test
8506,CG,5361EE8C3C1F32805CF7DCD382C3BBF132CEBF26,d391e75a9b981397e84339c885c864da_right,single,test
8519,CG,63597A7850B0966D88F466C755AB523C988F09B7,c987dafe0bbd44b4ee4de353d7ddeb32_left,single,test
8520,CG,63597A7850B0966D88F466C755AB523C988F09B7,c987dafe0bbd44b4ee4de353d7ddeb32_right,single,test
8560,CG,CD713E603F7655DDC1803C1A9DD7CB7E2728E765,93effacee0f8749285b35bacfdd9f2e6_left,single,test
...,...,...,...,...,...
20640,CG,563A9D53F301E21115802A01BBA930E6AB2AC5E2,7551ffe32b65208b2e52442ef55ed767_left,none,test
20641,CG,0A2439C3D54EC10A2C1B2BC5C4113ABF4536F35B,27f5efada664aa4858a8bced66b88b54,none,test
20642,CG,0A2439C3D54EC10A2C1B2BC5C4113ABF4536F35B,6fe5790e414a7f7cdd28c458e944a271,none,test
20643,CG,0A2439C3D54EC10A2C1B2BC5C4113ABF4536F35B,42eba5aa053b492b9e6f7bb781a058f3,none,test


In [44]:
print("single nodule image :", len(CG_data_df[CG_data_df["nodule_type"] == "single"]))
print("multiple nodule image :", len(CG_data_df[CG_data_df["nodule_type"] == "multiple"]))
print("none nodule image :", len(CG_data_df[CG_data_df["nodule_type"] == "none"]))

single nodule image : 1328
multiple nodule image : 116
none nodule image : 920


In [45]:
print("single nodule patient :", len(np.unique(CG_data_df[CG_data_df["nodule_type"] == "single"]["patient_ID"])))
print("multiple nodule patient :", len(np.unique(CG_data_df[CG_data_df["nodule_type"] == "multiple"]["patient_ID"])))
print("none nodule patient :", len(np.unique(CG_data_df[CG_data_df["nodule_type"] == "none"]["patient_ID"])))

single nodule patient : 404
multiple nodule patient : 61
none nodule patient : 211


## 有結節影像：平均每張結節數

In [46]:
label_dir = 'yolo_data/all_data_nodule_normal_cut_od/labels'

def count_nodules(fname):
    with open(f'{label_dir}/{fname}.txt') as f:
        return sum(1 for line in f if line.strip())

CG_data_df = CG_data_df.copy()
CG_data_df['nodule_count'] = CG_data_df['file_name'].apply(count_nodules)

nodule_imgs = CG_data_df[CG_data_df['nodule_type'] != 'none']

mean_n = nodule_imgs['nodule_count'].mean()
print(f'有結節影像數: {len(nodule_imgs)}')
print(f'平均每張結節數: {mean_n:.2f}')
print()
print('結節數分布:')
dist = nodule_imgs['nodule_count'].value_counts().sort_index()
dist_df = pd.DataFrame({'結節數': dist.index, '影像張數': dist.values,
                         '比例 (%)': (dist.values / dist.values.sum() * 100).round(1)})
display(dist_df)

有結節影像數: 1444
平均每張結節數: 1.25

結節數分布:


,結節數,影像張數,比例 (%)
0,1,1163,80.5
1,2,215,14.9
2,3,55,3.8
3,4,4,0.3
4,5,6,0.4
5,6,1,0.1


In [47]:
print("total image :", len(CG_data_df))
print("total patient :", len(np.unique(CG_data_df["patient_ID"])))

total image : 2364
total patient : 617


# CG 病歷報告統計分析（all_reports）

In [48]:
import pandas as pd
import numpy as np
import os

# ── 讀取所有病歷報告 ──────────────────────────────────────────────
BASE = '../thyroid_old/data/CG_data/all_reports'

dfs = []
for f in sorted(os.listdir(BASE)):
    if not f.endswith('.xlsx'):
        continue
    path = os.path.join(BASE, f)
    try:
        df = pd.read_excel(path, engine='calamine')
    except Exception:
        df = pd.read_excel(path, engine='openpyxl')
    dfs.append(df)

all_df = pd.concat(dfs, ignore_index=True)

# ── 篩選 CG test 病人 ─────────────────────────────────────────────
csv_df = pd.read_csv('yolo_data/all_csv_files/combined_dataset_info_v5.csv')
cg_test_patients = csv_df[
    (csv_df['dataset'] == 'CG') & (csv_df['train_or_test'] == 'test')
]['patient_ID'].unique()

filtered = all_df[all_df['IDCODE'].isin(cg_test_patients)].copy()

# EDATE 格式不一，統一轉 datetime
filtered['exam_date'] = pd.to_datetime(filtered['EDATE'], format='mixed', dayfirst=False)

# 每位病人取最早一筆（age 以第一次檢查為準）
patient_df = (filtered
              .sort_values('exam_date')
              .drop_duplicates(subset='IDCODE', keep='first')
              .reset_index(drop=True))

print(f'CG test 病人數 : {len(cg_test_patients)}')
print(f'對回報告後 (dedup): {len(patient_df)} 位病人')

CG test 病人數 : 617
對回報告後 (dedup): 617 位病人


## 1. 資料時間範圍

In [49]:
date_min = patient_df['exam_date'].min()
date_max = patient_df['exam_date'].max()

print(f'資料時間範圍: {date_min.strftime("%Y-%m-%d")} ~ {date_max.strftime("%Y-%m-%d")}')
print(f'共橫跨 {(date_max - date_min).days} 天')

資料時間範圍: 1970-01-01 ~ 2022-12-29
共橫跨 19354 天


## 2. 性別分布

In [50]:
sex_counts = patient_df['SEX'].value_counts()
sex_pct = patient_df['SEX'].value_counts(normalize=True) * 100

sex_summary = pd.DataFrame({
    '人數': sex_counts,
    '比例 (%)': sex_pct.round(1)
})
sex_summary.index.name = '性別'
sex_summary.index = sex_summary.index.map({'F': '女 (F)', 'M': '男 (M)'})

print('性別分布:')
display(sex_summary)
print(f'\n總計: {sex_counts.sum()} 人')

性別分布:


,人數,比例 (%)
性別,,
女 (F),491,79.7
男 (M),125,20.3



總計: 616 人


## 3. 年齡（平均 ± SD）

In [29]:
age = patient_df['age'].dropna()

age_mean = age.mean()
age_sd   = age.std()
age_min  = age.min()
age_max  = age.max()
age_med  = age.median()

print(f'年齡（平均 ± SD）: {age_mean:.1f} ± {age_sd:.1f} 歲')
print(f'中位數: {age_med:.1f} 歲')
print(f'範圍: {age_min:.1f} ~ {age_max:.1f} 歲')
print(f'有效人數: {len(age)}  缺值: {patient_df["age"].isna().sum()}')

年齡（平均 ± SD）: 47.4 ± 15.6 歲
中位數: 47.5 歲
範圍: 0.0 ~ 103.3 歲
有效人數: 3191  缺值: 1


## 4. 綜合摘要表

In [30]:
summary = {
    '病人總數':       len(patient_df),
    '資料起始日':     date_min.strftime('%Y-%m-%d'),
    '資料結束日':     date_max.strftime('%Y-%m-%d'),
    '女性人數 (%)':   f"{sex_counts.get('F', 0)} ({sex_pct.get('F', 0):.1f}%)",
    '男性人數 (%)':   f"{sex_counts.get('M', 0)} ({sex_pct.get('M', 0):.1f}%)",
    '年齡平均 ± SD':  f'{age_mean:.1f} ± {age_sd:.1f}',
    '年齡中位數':     f'{age_med:.1f}',
    '年齡範圍':       f'{age_min:.1f} ~ {age_max:.1f}',
}

summary_df = pd.DataFrame(summary.items(), columns=['項目', '數值'])
display(summary_df)

,項目,數值
0,病人總數,3192
1,資料起始日,1970-01-01
2,資料結束日,2022-12-29
3,女性人數 (%),2480 (77.7%)
4,男性人數 (%),711 (22.3%)
5,年齡平均 ± SD,47.4 ± 15.6
6,年齡中位數,47.5
7,年齡範圍,0.0 ~ 103.3


# 分層分析（Subgroup Analysis）

In [52]:
import cv2, re

label_dir = 'yolo_data/all_data_nodule_normal_cut_od/labels'
image_dir = 'yolo_data/all_data_nodule_normal_cut_od/images'

csv_df = pd.read_csv('yolo_data/all_csv_files/combined_dataset_info_v5.csv')
cg = csv_df[(csv_df['dataset'] == 'CG') & (csv_df['train_or_test'] == 'test')].copy()
nodule_cg = cg[cg['nodule_type'] != 'none'].copy()

# 從 label 檔逐顆結節建立 DataFrame
records = []
for _, row in nodule_cg.iterrows():
    fname = row['file_name']
    img_path = f'{image_dir}/{fname}.png'
    if not os.path.exists(img_path):
        img_path = f'{image_dir}/{fname}.jpg'
    img = cv2.imread(img_path)
    if img is None:
        continue
    H, W = img.shape[:2]
    with open(f'{label_dir}/{fname}.txt') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            _, cx, cy, w, h = int(parts[0]), float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
            records.append({
                'file_name':     fname,
                'patient_ID':    row['patient_ID'],
                'nodule_type':   row['nodule_type'],
                'train_or_test': row['train_or_test'],
                'cx': cx, 'cy': cy,
                'w_px': w * W, 'h_px': h * H,
                'max_dim_px': max(w * W, h * H),
            })

nodule_df = pd.DataFrame(records)
print(f'CG test 結節 instances 總數: {len(nodule_df)}')

CG test 結節 instances 總數: 1810


## 1. 結節大小分層（依 max dimension 像素分三群）

> 無 pixel spacing，以像素最大邊長分三等份（小 / 中 / 大）作為相對大小代理指標。

In [53]:
q33 = nodule_df['max_dim_px'].quantile(1/3)
q67 = nodule_df['max_dim_px'].quantile(2/3)

nodule_df['size_group'] = pd.cut(
    nodule_df['max_dim_px'],
    bins=[0, q33, q67, np.inf],
    labels=[f'小 (<{q33:.0f}px)', f'中 ({q33:.0f}–{q67:.0f}px)', f'大 (≥{q67:.0f}px)']
)

size_stats = nodule_df.groupby('size_group', observed=True).agg(
    結節數=('max_dim_px', 'count'),
    平均最大邊長_px=('max_dim_px', 'mean'),
    中位最大邊長_px=('max_dim_px', 'median'),
    平均寬_px=('w_px', 'mean'),
    平均高_px=('h_px', 'mean'),
).round(1)
size_stats['比例 (%)'] = (size_stats['結節數'] / size_stats['結節數'].sum() * 100).round(1)

print(f'三分位切點: {q33:.1f} px / {q67:.1f} px\n')
display(size_stats)

三分位切點: 62.0 px / 114.0 px



,結節數,平均最大邊長_px,中位最大邊長_px,平均寬_px,平均高_px,比例 (%)
size_group,,,,,,
小 (<62px),610,44.2,45.0,43.9,31.4,33.7
中 (62–114px),598,85.2,84.0,84.5,59.0,33.0
大 (≥114px),602,199.8,167.0,198.9,127.8,33.3


## 2. 結節位置分層（左葉 / 右葉）與 View（Saggital / Transverse）

In [54]:
info_all = pd.read_excel('../thyroid_old/data/CG_data/all_csv_files/data_info_nodule.xlsx')
info_all['position'] = info_all['position'].str.strip().replace('lright', 'right')
info_all['view'] = info_all['view'].str.strip()

# 只保留 CG test 的影像
test_files = csv_df[(csv_df['dataset']=='CG') & (csv_df['train_or_test']=='test')]['file_name']
info = info_all[info_all['file_name'].isin(test_files)].copy()

# ── 位置分布 ────────────────────────────────────────────────────
pos_counts = info['position'].value_counts()
pos_pct    = info['position'].value_counts(normalize=True) * 100

pos_df = pd.DataFrame({
    '影像數':   pos_counts,
    '比例 (%)': pos_pct.round(1),
})
pos_df.index = pos_df.index.map({'left': '左葉', 'right': '右葉'})
pos_df.index.name = '位置'

print('【位置分布】')
display(pos_df)

# ── View 分布 ──────────────────────────────────────────────────
view_counts = info['view'].value_counts()
view_pct    = info['view'].value_counts(normalize=True) * 100

view_df = pd.DataFrame({
    '影像數':   view_counts,
    '比例 (%)': view_pct.round(1),
})
view_df.index.name = 'View'

print('\n【View 分布】')
display(view_df)

# ── 位置 × View 交叉表 ─────────────────────────────────────────
cross_pv = pd.crosstab(
    info['position'].map({'left': '左葉', 'right': '右葉'}),
    info['view'],
    margins=True, margins_name='合計'
)
cross_pv.index.name = '位置'

print('\n【位置 × View 交叉表】')
display(cross_pv)

【位置分布】


,影像數,比例 (%)
位置,,
右葉,745,51.6
左葉,699,48.4



【View 分布】


,影像數,比例 (%)
View,,
saggital,739,51.2
transverse,705,48.8



【位置 × View 交叉表】


view,saggital,transverse,合計
位置,,,
右葉,389,356,745
左葉,350,349,699
合計,739,705,1444


## 3. 回音特性分層（從 ABSTRACT 關鍵字萃取）

> 覆蓋率有限（僅部分報告有文字描述），以下為有描述的子集分析。

In [55]:
BASE = '../thyroid_old/data/CG_data/all_reports'
dfs_r = []
for f in sorted(os.listdir(BASE)):
    if not f.endswith('.xlsx'):
        continue
    path = os.path.join(BASE, f)
    try:
        df = pd.read_excel(path, engine='calamine')
    except Exception:
        df = pd.read_excel(path, engine='openpyxl')
    dfs_r.append(df)
all_rep = pd.concat(dfs_r, ignore_index=True)

# 只取 CG test 病人
cg_rep = all_rep[all_rep['IDCODE'].isin(cg_test_patients)][
    ['IDCODE', 'ABSTRACT', 'REPORT01', 'positive']
].copy()

# 從 ABSTRACT 及 REPORT01 萃取回音特性關鍵字
text_col = cg_rep['ABSTRACT'].fillna('') + ' ' + cg_rep['REPORT01'].fillna('')

echo_patterns = {
    'hypoechoic':       r'hypo.?echo',
    'hyperechoic':      r'hyper.?echo',
    'isoechoic':        r'iso.?echo',
    'cystic/mixed':     r'cystic|mixed echo',
    'calcification':    r'calcif',
    'ill-defined':      r'ill.?defined',
    'taller-than-wide': r'taller.than.wide|taller than wide',
    'halo':             r'\bhalo\b',
}

for label, pattern in echo_patterns.items():
    cg_rep[label] = text_col.str.contains(pattern, case=False, na=False).astype(int)

echo_cols = list(echo_patterns.keys())
echo_summary = pd.DataFrame({
    '特性':       echo_cols,
    '報告數':     [cg_rep[c].sum() for c in echo_cols],
    '覆蓋率 (%)': [(cg_rep[c].sum() / len(cg_rep) * 100) for c in echo_cols],
}).set_index('特性').round(1)

print(f'CG test 病人病歷報告數: {len(cg_rep)}\n')
display(echo_summary)

CG test 病人病歷報告數: 651



,報告數,覆蓋率 (%)
特性,,
hypoechoic,6,0.9
hyperechoic,0,0.0
isoechoic,2,0.3
cystic/mixed,1,0.2
calcification,1,0.2
ill-defined,1,0.2
taller-than-wide,0,0.0
halo,2,0.3


## 4. 惡性程度分層（`positive` 欄：0 = 良性 / 1 = 惡性）

In [56]:
pos_map = {0.0: '良性 (0)', 1.0: '惡性 (1)'}
cg_rep['malignancy'] = cg_rep['positive'].map(pos_map)

mal_counts = cg_rep['malignancy'].value_counts(dropna=False)
mal_pct = cg_rep['malignancy'].value_counts(normalize=True, dropna=False) * 100

mal_df = pd.DataFrame({
    '報告數': mal_counts,
    '比例 (%)': mal_pct.round(1),
})
mal_df.index.name = '惡性程度'

print(f'有 positive 標記筆數: {cg_rep["positive"].notna().sum()} / {len(cg_rep)}\n')
display(mal_df)

# 各子群大小分層 × 惡性程度（僅有 positive 標記者）
print('\n--- 有標記病例中回音特性分布 ---')
labeled = cg_rep[cg_rep['positive'].notna()].copy()
labeled['malignancy'] = labeled['positive'].map({0.0: '良性', 1.0: '惡性'})
echo_by_mal = labeled.groupby('malignancy')[echo_cols].sum()
display(echo_by_mal)

有 positive 標記筆數: 533 / 651



,報告數,比例 (%)
惡性程度,,
良性 (0),311,47.8
惡性 (1),222,34.1
NaN,118,18.1



--- 有標記病例中回音特性分布 ---


,hypoechoic,hyperechoic,isoechoic,cystic/mixed,calcification,ill-defined,taller-than-wide,halo
malignancy,,,,,,,,
惡性,0,0,0,0,0,0,0,0
良性,0,0,0,0,0,0,0,0


## 5. 綜合：位置 × 大小 × Single/Multiple 交叉表

In [57]:
# 用 data_info_nodule 的 position 取代 filename 猜測
nodule_df = nodule_df.merge(
    info[['file_name', 'position', 'view']].rename(columns={'position': 'pos_info', 'view': 'view_info'}),
    on='file_name', how='left'
)
nodule_df['pos_info'] = nodule_df['pos_info'].str.strip().replace('lright', 'right')
nodule_df['loc_label'] = nodule_df['pos_info'].map({'left': '左葉', 'right': '右葉'}).fillna('不明')

cross = pd.crosstab(
    index=[nodule_df['loc_label'], nodule_df['size_group']],
    columns=nodule_df['nodule_type'],
    margins=True, margins_name='合計'
)
cross.index.names = ['位置', '大小']
display(cross)

nodule_type      multiple  single    合計
位置 大小                                  
右葉 小 (<62px)           61     264   325
   中 (62–114px)        64     257   321
   大 (≥114px)          38     270   308
左葉 小 (<62px)           25     260   285
   中 (62–114px)        44     233   277
   大 (≥114px)          30     264   294
合計                    262    1548  1810